In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns

Step 1: Load the data

In [8]:
# loading data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

Step 2: Drop irrelevant coloumns
- PassengerId and Name do not carry an predictive info -> DROP
- Cabin encodes deck, number and side; split later
- keep the other coloumns  

In [9]:
# before dropping Cabin, split it first!
train[['CabinDeck', 'CabinNum', 'Side']] = train['Cabin'].str.split('/', expand=True)
test[['CabinDeck', 'CabinNum', 'Side']] = test['Cabin'].str.split('/', expand=True)

# convert CabinNum to number
train['CabinNum'] = pd.to_numeric(train['CabinNum'], errors='coerce')
test['CabinNum'] = pd.to_numeric(test['CabinNum'], errors='coerce')

# dropping 
train = train.drop(['PassengerId', 'Name', 'Cabin'], axis=1)
test = test.drop(['PassengerId', 'Name', 'Cabin'], axis=1)

Step 3: Handling missing values

In [10]:
# numerical col
num_cols = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','CabinNum']
for col in num_cols: 
    median_val = train[col].median()
    train[col].fillna(median_val,inplace=True)
    test[col].fillna(median_val,inplace=True)

# categ/bool col
categ_col = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
for col in categ_col: 
    mode_value = train[col].mode()[0]
    train[col].fillna(mode_value,inplace=True)
    test[col].fillna(mode_value,inplace=True)
    

Step 4: Encode categorical variables
- converting to numerical 

In [11]:
from sklearn.preprocessing import LabelEncoder

le_home = LabelEncoder()
train['HomePlanet'] = le_home.fit_transform(train['HomePlanet'])
test['HomePlanet'] = le_home.transform(test['HomePlanet'])

le_dest = LabelEncoder()
train['Destination'] = le_dest.fit_transform(train['Destination'])
test['Destination'] = le_dest.transform(test['Destination'])

le_cryosleep = LabelEncoder()
train['CryoSleep'] = le_cryosleep.fit_transform(train['CryoSleep'])
test['CryoSleep'] = le_cryosleep.transform(test['CryoSleep'])

le_vip = LabelEncoder()
train['VIP'] = le_vip.fit_transform(train['VIP'])
test['VIP'] = le_vip.transform(test['VIP'])

le_deck = LabelEncoder()
train['CabinDeck'] = le_deck.fit_transform(train['CabinDeck'].astype(str))
test['CabinDeck'] = le_deck.transform(test['CabinDeck'].astype(str))

le_side = LabelEncoder()
train['Side'] = le_side.fit_transform(train['Side'].astype(str))
test['Side'] = le_side.transform(test['Side'].astype(str))

Step 5 & 6: Split data into features and Build the NN model

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

features = ['HomePlanet', 'CryoSleep', 'CabinDeck', 'CabinNum', 'Side', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

result_dict = []

for feature in features:
   
    X = train[[feature]]
    y = train['Transported']

    
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # initialize and train neural network
    nn = MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=42)
    nn.fit(X_train_scaled, y_train)

    # predict on validation set
    y_pred = nn.predict(X_val_scaled)

    # Evaluate
    result_dict.append({'Accuracy': accuracy_score(y_val, y_pred),
                        'F1 Score': (f1_score(y_val, y_pred)),
                        'Confusion Matrix': confusion_matrix(y_val, y_pred)
                    })
    

In [16]:
sorted_list = sorted(result_dict, key=lambda x: x['Accuracy'], reverse=True)

print(sorted_list)

[{'Accuracy': 0.7222541690626797, 'F1 Score': 0.6811881188118811, 'Confusion Matrix': array([[740, 121],
       [362, 516]], dtype=int64)}, {'Accuracy': 0.6440483036227717, 'F1 Score': 0.7220475976650202, 'Confusion Matrix': array([[316, 545],
       [ 74, 804]], dtype=int64)}, {'Accuracy': 0.6319723979298447, 'F1 Score': 0.711971197119712, 'Confusion Matrix': array([[308, 553],
       [ 87, 791]], dtype=int64)}, {'Accuracy': 0.6308223116733755, 'F1 Score': 0.7118491921005385, 'Confusion Matrix': array([[304, 557],
       [ 85, 793]], dtype=int64)}, {'Accuracy': 0.6175963197239793, 'F1 Score': 0.7016599371915657, 'Confusion Matrix': array([[292, 569],
       [ 96, 782]], dtype=int64)}, {'Accuracy': 0.5934445083381253, 'F1 Score': 0.6847971466785555, 'Confusion Matrix': array([[264, 597],
       [110, 768]], dtype=int64)}, {'Accuracy': 0.5807935595169638, 'F1 Score': 0.5642558278541542, 'Confusion Matrix': array([[538, 323],
       [406, 472]], dtype=int64)}, {'Accuracy': 0.557791834387

In [26]:

best_n = 5  # change this based on your results
best_features = features[:best_n]
print("Best features:", best_features)

X = train[best_features]
y = train['Transported']

    
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# initialize and train neural network
nn = MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=42)
nn.fit(X_train_scaled, y_train)

# predict on validation set
y_pred = nn.predict(X_val_scaled)

# Evaluate
print(f'Accuracy: {accuracy_score(y_val, y_pred)}')
print(f'F1 Score: {(f1_score(y_val, y_pred))}')
print(f'Confusion Matrix: {confusion_matrix(y_val, y_pred)}')
            

Best features: ['HomePlanet', 'CryoSleep', 'CabinDeck', 'CabinNum', 'Side']
Accuracy: 0.7423806785508913
F1 Score: 0.7192982456140352
Confusion Matrix: [[717 144]
 [304 574]]
